In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [0]:
ride_schema=StructType([StructField('ride_id', StringType(), True), StructField('confirmation_number', StringType(), True), StructField('passenger_id', StringType(), True), StructField('driver_id', StringType(), True), StructField('vehicle_id', StringType(), True), StructField('pickup_location_id', StringType(), True), StructField('dropoff_location_id', StringType(), True), StructField('vehicle_type_id', LongType(), True), StructField('vehicle_make_id', LongType(), True), StructField('payment_method_id', LongType(), True), StructField('ride_status_id', LongType(), True), StructField('pickup_city_id', LongType(), True), StructField('dropoff_city_id', LongType(), True), StructField('cancellation_reason_id', LongType(), True), StructField('passenger_name', StringType(), True), StructField('passenger_email', StringType(), True), StructField('passenger_phone', StringType(), True), StructField('driver_name', StringType(), True), StructField('driver_rating', DoubleType(), True), StructField('driver_phone', StringType(), True), StructField('driver_license', StringType(), True), StructField('vehicle_model', StringType(), True), StructField('vehicle_color', StringType(), True), StructField('license_plate', StringType(), True), StructField('pickup_address', StringType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('dropoff_address', StringType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('distance_miles', DoubleType(), True), StructField('duration_minutes', LongType(), True), StructField('booking_timestamp', TimestampType(), True), StructField('pickup_timestamp', StringType(), True), StructField('dropoff_timestamp', StringType(), True), StructField('base_fare', DoubleType(), True), StructField('distance_fare', DoubleType(), True), StructField('time_fare', DoubleType(), True), StructField('surge_multiplier', DoubleType(), True), StructField('subtotal', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('total_fare', DoubleType(), True), StructField('rating', DoubleType(), True)])

In [0]:
df=spark.read.table('uber.bronze.rides_raw')
df_parse=df.withColumn("parse_rides",from_json(col("rides"),schema=ride_schema)).select("parse_rides.*")

display(df_parse)

In [0]:
df=spark.sql("select * from uber.bronze.bulk_rides")
df.schema

In [0]:
%sql select * from uber.bronze.stg_rides

In [0]:
from jinja2 import Template

jinja_config = [
    {
        "table": "uber.bronze.stg_rides",
        "select": "stg_rides.*",
        "where": ""
    },
    {
        "table": "uber.bronze.map_vehicle_makes",
        "select": "map_vehicle_makes.vehicle_make",
        "where": "",
        "on": "stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table": "uber.bronze.map_vehicle_types",
        "select": "map_vehicle_types.description, map_vehicle_types.base_rate, map_vehicle_types.per_mile, map_vehicle_types.per_minute",
        "where": "",
        "on": "stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
    }
]

template = Template("""
SELECT
    {{ config[0].select }}

    {% for item in config[1:] %}
    , {{ item.select }}
    {% endfor %}

FROM {{ config[0].table }}

{% for item in config[1:] %}
left JOIN {{ item.table }}
    ON {{ item.on }}
{% endfor %}

{% if config[0].where %}
WHERE {{ config[0].where }}
{% endif %}
""")

query = template.render(config=jinja_config)

print(query)

In [0]:
df=spark.sql(query)
display(df)

In [0]:
%sql select * from uber.bronze.silver_obt

**Testing**

In [0]:
%sql select * from uber.bronze.dim_location

In [0]:
%sql
select * from uber.bronze.dim_driver

In [0]:
%sql
select * from uber.bronze.dim_passenger

In [0]:
%sql select * from uber.bronze.fact f
left join uber.bronze.dim_location l
on f.pickup_city_id=l.pickup_city_id
where l.`__END_AT` is null